# Часть 6 из 8. Ансамбль CatBoost, LightGBM и XGBoost

**Проект:** прогноз цены квартиры на вторичном рынке Москвы по датасету `Moscow_Real_Estate`.

**В этой части:**
- три модели на 41 признаке;
- простое среднее, веса от Optuna и стекинг с Ridge на out-of-fold прогнозах.

**Главный результат:** CatBoost — 6,735 %, взвешенное среднее — 6,720 %, стекинг — 6,719 % MAPE на валидации.

> Код продолжает предыдущие части и использует их переменные. Результаты уже сохранены в блокноте, поэтому его можно читать без запуска. Чтобы пересчитать часть, сначала выполните в этом же ядре части 1–5 по порядку: `%run -i 01_eda.ipynb` и так далее.

← [Часть 5. Новые признаки и CatBoost на них](05_feature_engineering.ipynb) · [Все части](README.md) · [Часть 7. Эксперименты с нейросетями](07_neural_networks.ipynb) →

### Ансамбль CatBoost, LightGBM, XGBoost

In [70]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb

# Данные
TARGET = 'price_rub'
df_ens = df_cb.copy()

# date-подобные
bad = [c for c in df_ens.columns
       if isinstance(df_ens[c].dtype, pd.PeriodDtype)
       or pd.api.types.is_datetime64_any_dtype(df_ens[c])]
df_ens = df_ens.drop(columns=bad)

# Категориальные
cat_cols = [c for c in ['district', 'okrug', 'metro_station', 'metro_line',
                        'building_type', 'renovation']
            if c in df_ens.columns]

X = df_ens.drop(columns=[TARGET])
y = np.log1p(df_ens[TARGET].values)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f"train: {X_tr.shape}, val: {X_val.shape}, test: {X_test.shape}")

train: (35000, 41), val: (7500, 41), test: (7500, 41)


In [73]:
# Для CatBoost — категориальные как строки
X_tr_cb = X_tr.copy(); X_val_cb = X_val.copy()
for c in cat_cols:
    X_tr_cb[c]  = X_tr_cb[c].astype(str)
    X_val_cb[c] = X_val_cb[c].astype(str)

cat_idx = [X_tr_cb.columns.get_loc(c) for c in cat_cols]

model_cb = CatBoostRegressor(
    iterations=5000, learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
    loss_function='RMSE', eval_metric='MAPE',
    random_seed=42, early_stopping_rounds=200, verbose=500,
    cat_features=cat_idx,
)
t0 = time.time()
model_cb.fit(X_tr_cb, y_tr, eval_set=(X_val_cb, y_val), use_best_model=True)
print(f"CatBoost: {time.time()-t0:.1f} сек")

pred_cb_val = model_cb.predict(X_val_cb)

0:	learn: 0.0293358	test: 0.0290509	best: 0.0290509 (0)	total: 89.8ms	remaining: 7m 29s
500:	learn: 0.0042641	test: 0.0042695	best: 0.0042695 (500)	total: 48.5s	remaining: 7m 15s
1000:	learn: 0.0038871	test: 0.0041599	best: 0.0041598 (995)	total: 1m 36s	remaining: 6m 24s
1500:	learn: 0.0036346	test: 0.0041449	best: 0.0041448 (1498)	total: 2m 23s	remaining: 5m 34s
2000:	learn: 0.0034171	test: 0.0041401	best: 0.0041399 (1997)	total: 3m 10s	remaining: 4m 45s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.004139852652
bestIteration = 1997

Shrink model to first 1998 iterations.
CatBoost: 210.9 сек


In [74]:
# Для LightGBM — категориальные как 'category'
X_tr_lgb = X_tr.copy(); X_val_lgb = X_val.copy()
for c in cat_cols:
    X_tr_lgb[c]  = X_tr_lgb[c].astype('category')
    X_val_lgb[c] = X_val_lgb[c].astype('category')

model_lgb = lgb.LGBMRegressor(
    n_estimators=5000, learning_rate=0.03, num_leaves=63,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1,
)
t0 = time.time()
model_lgb.fit(
    X_tr_lgb, y_tr,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric='mape',
    callbacks=[lgb.early_stopping(200, verbose=False)],
)
print(f"LightGBM: {time.time()-t0:.1f} сек")

pred_lgb_val = model_lgb.predict(X_val_lgb)

c:\Users\Computer\Desktop\programInj\house-price-predictor-\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM: 2.3 сек


In [75]:
X_tr_xgb = X_tr.copy(); X_val_xgb = X_val.copy()
for c in cat_cols:
    X_tr_xgb[c]  = X_tr_xgb[c].astype('category')
    X_val_xgb[c] = X_val_xgb[c].astype('category')

model_xgb = xgb.XGBRegressor(
    n_estimators=5000, learning_rate=0.03, max_depth=8,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42,
    tree_method='hist', enable_categorical=True,
    early_stopping_rounds=200, verbosity=0,
)
t0 = time.time()
model_xgb.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)
print(f"XGBoost: {time.time()-t0:.1f} сек")

pred_xgb_val = model_xgb.predict(X_val_xgb)

XGBoost: 5.2 сек


In [76]:
def mape_from_log(pred_log, y_log):
    pred = np.expm1(pred_log); true = np.expm1(y_log)
    return np.mean(np.abs((pred - true) / true)) * 100

print("\n=== Одиночные модели ===")
print(f"CatBoost: {mape_from_log(pred_cb_val,  y_val):.3f}%")
print(f"LightGBM: {mape_from_log(pred_lgb_val, y_val):.3f}%")
print(f"XGBoost:  {mape_from_log(pred_xgb_val, y_val):.3f}%")


=== Одиночные модели ===
CatBoost: 6.735%
LightGBM: 7.223%
XGBoost:  8.226%


In [77]:
pred_avg_log = (pred_cb_val + pred_lgb_val + pred_xgb_val) / 3
print(f"\nПростое среднее: {mape_from_log(pred_avg_log, y_val):.3f}%")


Простое среднее: 7.042%


In [78]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(w_trial):
    w = np.array([w_trial.suggest_float(f'w{i}', 0, 1) for i in range(3)])
    w = w / w.sum()  # нормализуем в сумму 1
    pred_log = w[0]*pred_cb_val + w[1]*pred_lgb_val + w[2]*pred_xgb_val
    return mape_from_log(pred_log, y_val)

study_ens = optuna.create_study(direction='minimize')
study_ens.optimize(objective, n_trials=100)

w = np.array([study_ens.best_params[f'w{i}'] for i in range(3)])
w = w / w.sum()
print(f"\nЛучшие веса: CB={w[0]:.3f}, LGB={w[1]:.3f}, XGB={w[2]:.3f}")

pred_w_log = w[0]*pred_cb_val + w[1]*pred_lgb_val + w[2]*pred_xgb_val
print(f"Взвешенное среднее: {mape_from_log(pred_w_log, y_val):.3f}%")


Лучшие веса: CB=0.849, LGB=0.150, XGB=0.001
Взвешенное среднее: 6.720%


In [79]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

# OOF-предсказания на train — чтобы мета-модель не переобучилась
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_cb  = np.zeros(len(X_tr_cb))
oof_lgb = np.zeros(len(X_tr_lgb))
oof_xgb = np.zeros(len(X_tr_xgb))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tr_cb)):
    print(f"Fold {fold+1}/5")
    
    # CatBoost
    m = CatBoostRegressor(
        iterations=5000, learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
        loss_function='RMSE', random_seed=42, early_stopping_rounds=100,
        verbose=False, cat_features=cat_idx,
    )
    m.fit(X_tr_cb.iloc[tr_idx], y_tr[tr_idx],
          eval_set=(X_tr_cb.iloc[va_idx], y_tr[va_idx]), use_best_model=True)
    oof_cb[va_idx] = m.predict(X_tr_cb.iloc[va_idx])
    
    # LightGBM
    m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.03, num_leaves=63,
                          random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_tr_lgb.iloc[tr_idx], y_tr[tr_idx],
          eval_set=[(X_tr_lgb.iloc[va_idx], y_tr[va_idx])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[va_idx] = m.predict(X_tr_lgb.iloc[va_idx])
    
    # XGBoost
    m = xgb.XGBRegressor(n_estimators=5000, learning_rate=0.03, max_depth=8,
                         random_state=42, tree_method='hist',
                         enable_categorical=True, early_stopping_rounds=100,
                         verbosity=0)
    m.fit(X_tr_xgb.iloc[tr_idx], y_tr[tr_idx],
          eval_set=[(X_tr_xgb.iloc[va_idx], y_tr[va_idx])], verbose=False)
    oof_xgb[va_idx] = m.predict(X_tr_xgb.iloc[va_idx])

# Мета-модель
meta = Ridge(alpha=1.0)
meta.fit(np.column_stack([oof_cb, oof_lgb, oof_xgb]), y_tr)

print(f"\nВеса мета-модели:")
print(f"  CatBoost: {meta.coef_[0]:.3f}")
print(f"  LightGBM: {meta.coef_[1]:.3f}")
print(f"  XGBoost:  {meta.coef_[2]:.3f}")
print(f"  intercept: {meta.intercept_:.3f}")

# Оценка на val
pred_stack_val = meta.predict(np.column_stack([pred_cb_val, pred_lgb_val, pred_xgb_val]))
print(f"\nStacking: {mape_from_log(pred_stack_val, y_val):.3f}%")

Fold 1/5


c:\Users\Computer\Desktop\programInj\house-price-predictor-\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2/5


c:\Users\Computer\Desktop\programInj\house-price-predictor-\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 3/5


...

Fold 4/5


c:\Users\Computer\Desktop\programInj\house-price-predictor-\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5/5


c:\Users\Computer\Desktop\programInj\house-price-predictor-\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Веса мета-модели:
  CatBoost: 0.820
  LightGBM: 0.192
  XGBoost:  -0.009
  intercept: -0.044

Stacking: 6.719%


In [80]:
results = pd.DataFrame({
    'CatBoost':        mape_from_log(pred_cb_val, y_val),
    'LightGBM':        mape_from_log(pred_lgb_val, y_val),
    'XGBoost':         mape_from_log(pred_xgb_val, y_val),
    'Avg (3 models)':  mape_from_log(pred_avg_log, y_val),
    'Weighted (Optuna)': mape_from_log(pred_w_log, y_val),
    'Stacking (Ridge)':  mape_from_log(pred_stack_val, y_val),
}, index=['MAPE %']).T

print(results.round(3))

                   MAPE %
CatBoost            6.735
LightGBM            7.223
XGBoost             8.226
Avg (3 models)      7.042
Weighted (Optuna)   6.720
Stacking (Ridge)    6.719


---
← [Часть 5. Новые признаки и CatBoost на них](05_feature_engineering.ipynb) · [Все части](README.md) · [Часть 7. Эксперименты с нейросетями](07_neural_networks.ipynb) →